In [4]:
import numpy as np

# LSTM FROM SCRATCH - TRADUCTION FR -> EN

# Jeu de données
# Chaque élément contient :
# (phrase française, traduction anglaise)
dataset = [
    ("bonjour", "hello"),
    ("merci", "thanks"),
    ("oui", "yes"),
    ("non", "no"),
    ("bonne nuit", "good night"),
    ("je mange", "i eat"),
    ("je cours", "i run"),
    ("je dors", "i sleep"),
    ("chat noir", "black cat"),
    ("chien blanc", "white dog")
]

# On récupère tous les mots français et anglais
# set permet d'éviter les doublons
fr_words = set()
en_words = set()

for fr, en in dataset:
    fr_words.update(fr.split())
    en_words.update(en.split())

# On trie les mots pour avoir toujours le même ordre
fr_words = sorted(list(fr_words))
en_words = sorted(list(en_words))

# Chaque mot reçoit un indice
fr_word_to_index = {
    word: i
    for i, word in enumerate(fr_words)
}

en_word_to_index = {
    word: i
    for i, word in enumerate(en_words)
}

# Pour retrouver le mot anglais à partir de son indice
index_to_en_word = {
    i: word
    for word, i in en_word_to_index.items()
}

input_size = len(fr_words)
output_size = len(en_words)

print("Nombre de mots français :", input_size)
print("Nombre de mots anglais  :", output_size)


# Fonction : one_hot()
#
# Cette fonction transforme un mot en vecteur One-Hot.
# Exemple :
# Vocabulaire :
# bonjour -> 0
# merci   -> 1
# oui     -> 2
#
# "merci" devient : [0 1 0]

def one_hot(index, vocab_size):

    vector = np.zeros((vocab_size, 1))
    vector[index] = 1

    return vector


# nombre de neurones dans l'état caché
hidden_size = 16


# Porte d'oubli
# elle décide quelle partie de l'ancienne mémoire on garde

Wf = np.random.randn(
    hidden_size,
    input_size + hidden_size
) * 0.01

bf = np.zeros((hidden_size, 1))


# Porte d'entrée
# elle décide quelles nouvelles informations on ajoute

Wi = np.random.randn(
    hidden_size,
    input_size + hidden_size
) * 0.01

bi = np.zeros((hidden_size, 1))


# Nouvelle information candidate
# c'est l'information qu'on peut ajouter dans la mémoire

Wc = np.random.randn(
    hidden_size,
    input_size + hidden_size
) * 0.01

bc = np.zeros((hidden_size, 1))


# Porte de sortie
# elle décide quelle partie de la mémoire devient la sortie

Wo = np.random.randn(
    hidden_size,
    input_size + hidden_size
) * 0.01

bo = np.zeros((hidden_size, 1))


# Couche de sortie : permet de choisir le mot anglais

Wy = np.random.randn(
    output_size,
    hidden_size
) * 0.01

by = np.zeros((output_size, 1))


# sigmoid donne des valeurs entre 0 et 1
# elle est utilisée pour les trois portes

def sigmoid(x):
    return 1 / (1 + np.exp(-x))


# tanh donne des valeurs entre -1 et 1
def tanh(x):
    return np.tanh(x)


# transforme les scores en probabilités
def softmax(x):

    # on enlève le maximum pour éviter les problèmes numériques
    x = x - np.max(x)

    exp = np.exp(x)

    return exp / np.sum(exp)


# cellule LSTM
# x_t : entrée actuelle
# h_prev : état caché précédent
# C_prev : ancienne mémoire

def lstm_cell(x_t, h_prev, C_prev):

    # On met ensemble l'entrée actuelle et l'état caché précédent
    combined = np.vstack((x_t, h_prev))


    # 1) porte d'oubli
    # décide ce qu'on oublie de C_prev
    f_t = sigmoid(
        np.dot(Wf, combined) + bf
    )


    # 2) porte d'entrée
    # décide quelles nouvelles informations sont importantes
    i_t = sigmoid(
        np.dot(Wi, combined) + bi
    )


    # nouvelle information qu'on pourrait ajouter à la mémoire
    C_candidate = tanh(
        np.dot(Wc, combined) + bc
    )


    # 3) mise à jour de la mémoire
    # on garde une partie de l'ancienne mémoire
    # et on ajoute une partie de la nouvelle information
    C_t = (
        f_t * C_prev
        +
        i_t * C_candidate
    )


    # 4) porte de sortie
    # décide ce qu'on va utiliser comme sortie
    o_t = sigmoid(
        np.dot(Wo, combined) + bo
    )


    # nouvel état caché
    # C_t = mémoire à long terme
    # h_t = mémoire courte / sortie actuelle
    h_t = o_t * tanh(C_t)


    # calcul de la sortie finale
    y = softmax(
        np.dot(Wy, h_t) + by
    )

    return (
        h_t,
        C_t,
        y,
        f_t,
        i_t,
        C_candidate,
        o_t
    )
    
train()

predict("bonjour")

predict("merci")

predict("chat noir")

Nombre de mots français : 14
Nombre de mots anglais  : 14
Début de l'entraînement du LSTM
Epoch 100/500 | Loss = 41.2252
Epoch 200/500 | Loss = 31.6038
Epoch 300/500 | Loss = 11.6506
Epoch 400/500 | Loss = 1.6567
Epoch 500/500 | Loss = 0.6090

Entraînement terminé.
Test du réseau
Français : bonjour
Traduction prédite : hello
Test du réseau
Français : merci
Traduction prédite : thanks
Test du réseau
Français : chat noir
Traduction prédite : black cat
